# Frozen fine-tuned Swin detector: train Swin ReID
Run this notebook twice, once per STRATEGY, to obtain separately trained models. Only ReID parameters are optimized. The verified detector artifact defaults to the final evaluation notebook's `swin-detector-512x768-best:v0`, run `77d9acgi`. Change to a local path if desired.

`crop_to_scene`: frozen scene/crop Swin+FPN and a fresh shared RoI ReID head; cross-image query-crop to GT-scene InfoNCE. Single scene backbone pass at gallery inference. `crop_to_crop`: independent ImageNet Swin-T, last two stages plus norm and global/upper/lower projection trainable, GT crop-to-crop InfoNCE. Detector remains independent and frozen. These are different training systems, not simply different inference modes.

50 fresh epochs, random P16/K4, 256x128 crops, temperature .1. Same saved detector resolution and split seed. Select best by fixed validation gallery retrieval mAP; evaluate official test after training. No detector loss, classification loss or hard mining. Use the same score threshold for all final comparisons.


In [ ]:
from kaggle_secrets import UserSecretsClient
import base64
import os
import subprocess
import sys
from pathlib import Path

USER = "SaadQAbbasi"
REPO = "ML4CV-Assignment"
repo_path = f"/kaggle/working/{REPO}"
repo_dir = Path(repo_path)
remote = f"https://github.com/{USER}/{REPO}.git"

# Pass credentials through the child environment, not the URL or notebook output.
token = UserSecretsClient().get_secret("github_token")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
})
try:
    if not repo_dir.exists():
        command = ["git", "clone", remote, str(repo_dir)]
    elif (repo_dir / ".git").is_dir():
        command = ["git", "-C", str(repo_dir), "pull", "--ff-only", remote]
    else:
        raise RuntimeError(f"{repo_dir} exists but is not a Git checkout.")
    result = subprocess.run(command, env=git_env, capture_output=True, text=True)
    if result.returncode:
        detail = (result.stderr + result.stdout).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
        raise RuntimeError(
            "Repository setup failed. Check Kaggle Internet access, GITHUB_TOKEN "
            "secret access for this notebook, and repository permissions.\n" + detail
        )
finally:
    del token, auth, git_env

src_dir = repo_dir / "src"
required = ("model.py", "prw_dataset.py", "training.py", "prw_evaluation.py", "dino_reid.py", "dinov3_reid.py", "offline_mining.py")
missing = [name for name in required if not (src_dir / name).is_file()]
if missing:
    raise RuntimeError(f"Repository at {repo_dir} is missing source files: {missing}")
src_path = str(src_dir.resolve())
if src_path in sys.path:
    sys.path.remove(src_path)
sys.path.insert(0, src_path)
import importlib
importlib.invalidate_caches()
print(f"Project imports configured from {src_path}")


In [ ]:


from kaggle_secrets import UserSecretsClient
import wandb

secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WANDB_API_KEY")

wandb.login(key=wandb_api_key)




PRW_ROOT = "/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild"

data_root = PRW_ROOT



In [ ]:
import random
import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from model import PersonSearchModel
from dino_reid import DinoPersonSearch
from swin_reid import SwinCropReID, SwinSceneReID, swin_reid_epoch
from prw_dataset import PRWPersonSearchDataset, PRWSubset, PKIdentityBatchSampler, prw_collate_fn
from training import make_grad_scaler, start_run
from validation_retrieval import build_validation_gallery, evaluate_validation_gallery
from prw_evaluation import evaluate_prw, make_wandb_retrieval_table, plot_retrieval_query

STRATEGY = 'crop_to_scene'  # alternatively 'crop_to_crop'; start a separate run
DETECTOR_LOCAL_PATH = None
DETECTOR_RUN_ID = '77d9acgi'
DETECTOR_ARTIFACT = 'swin-detector-512x768-best:v0'
ENTITY='models-alma-mater-studio'
PROJECT='person-search'
P,K=16,4
NUM_EPOCHS=50
SCORE_THRESHOLD=.5
TEMPERATURE=.1
CROP_SIZE=(256,128)
NUM_WORKERS=2
if STRATEGY not in ('crop_to_scene','crop_to_crop'):raise ValueError('Invalid strategy')
if DETECTOR_LOCAL_PATH is None:
    source=wandb.Api().run(f'{ENTITY}/{PROJECT}/{DETECTOR_RUN_ID}')
    matches=[a for a in source.logged_artifacts() if a.type=='model' and a.name.rsplit('/',1)[-1]==DETECTOR_ARTIFACT]
    if len(matches)!=1:raise RuntimeError('Verify detector run ID and artifact version')
    folder=Path('/kaggle/working/reid_detector')/DETECTOR_RUN_ID/DETECTOR_ARTIFACT.replace(':','-')
    detector_path=Path(matches[0].download(root=str(folder)))/'best.pt'
else:detector_path=Path(DETECTOR_LOCAL_PATH)
checkpoint=torch.load(detector_path,map_location='cpu',weights_only=False)
detector_config=dict(checkpoint['model_config'])
if detector_config['backbone_name']!='swin_t':raise ValueError('Expected Swin detector')
detector_config.update(pretrained=False,detection_only=True)
IMAGE_SIZE=tuple(checkpoint.get('detector_image_size',checkpoint['run_config']['image_size']))
SEED=int(checkpoint['run_config']['seed'])
VAL_FRACTION=float(checkpoint['run_config'].get('val_fraction',.1))
random.seed(SEED);torch.manual_seed(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detector=PersonSearchModel(**detector_config)
detector.load_state_dict(checkpoint['model_state_dict'],strict=True)
detector_epoch=checkpoint['epoch']
del checkpoint
if STRATEGY=='crop_to_scene':
    model=SwinSceneReID(detector,crop_size=CROP_SIZE,feature_batch_size=2)
else:
    model=DinoPersonSearch(detector,SwinCropReID(pretrained=True,crop_size=CROP_SIZE))
model=model.to(device)
GALLERY_MODE='scene' if STRATEGY=='crop_to_scene' else 'crop'
full=PRWPersonSearchDataset(PRW_ROOT,split='train',image_size=IMAGE_SIZE,query_crop_size=CROP_SIZE)
indices=torch.randperm(len(full),generator=torch.Generator().manual_seed(SEED)).tolist()
cut=round(len(indices)*VAL_FRACTION)
train_data=PRWSubset(full,indices[cut:]);val_data=PRWSubset(full,indices[:cut])
train_sampler=PKIdentityBatchSampler(train_data,p=P,k=K)
val_sampler=PKIdentityBatchSampler(val_data,p=4,k=2,seed=SEED)
train_loader=DataLoader(train_data,batch_sampler=train_sampler,collate_fn=prw_collate_fn,num_workers=NUM_WORKERS)
val_loader=DataLoader(val_data,batch_sampler=val_sampler,collate_fn=prw_collate_fn,num_workers=NUM_WORKERS)
# No detector parameter is present in the optimizer.
if STRATEGY=='crop_to_scene':
    groups=[{'params':model.reid_head.parameters(),'lr':1e-4}]
else:
    groups=[{'params':[p for n,p in model.reid_head.named_parameters() if p.requires_grad and not n.startswith('embedding_proj.')],'lr':1e-5},
            {'params':model.reid_head.embedding_proj.parameters(),'lr':1e-4}]
optimizer=torch.optim.AdamW(groups,weight_decay=1e-4)
scaler=make_grad_scaler(device)
assert not any(p.requires_grad for p in model.detector.parameters())
config=dict(strategy=STRATEGY,seed=SEED,val_fraction=VAL_FRACTION,P=P,K=K,epochs=NUM_EPOCHS,
            crop_size=CROP_SIZE,image_size=IMAGE_SIZE,temperature=TEMPERATURE,
            detector_path=str(detector_path),detector_artifact=DETECTOR_ARTIFACT,detector_epoch=detector_epoch,
            score_threshold=SCORE_THRESHOLD,checkpoint_selection='validation_retrieval_mAP')
run=start_run(config,name=f'swin-reid-{STRATEGY}-p16-k4',mode='online')
cache=build_validation_gallery(model,PRW_ROOT,[full.image_names[i] for i in val_data.indices],device,
                               score_threshold=SCORE_THRESHOLD,detector_image_size=IMAGE_SIZE)
checkpoint_dir=Path(repo_path)/f'checkpoints_swin_reid_{STRATEGY}'
checkpoint_dir.mkdir(parents=True,exist_ok=True)
best=-1.


In [ ]:
for epoch in range(1,NUM_EPOCHS+1):
    losses=swin_reid_epoch(model,train_loader,device,STRATEGY,optimizer,scaler,TEMPERATURE)
    validation_loss=swin_reid_epoch(model,val_loader,device,STRATEGY,temperature=TEMPERATURE)
    metrics=evaluate_validation_gallery(model,PRW_ROOT,cache,device,gallery_embedding_mode=GALLERY_MODE)
    run.log({'epoch':epoch,**{f'train/{k}':v for k,v in losses.items()},
             **{f'val/{k}':v for k,v in {**validation_loss,**metrics}.items()}})
    improved=metrics['retrieval_mAP']>best
    best=max(best,metrics['retrieval_mAP'])
    state=dict(epoch=epoch,strategy=STRATEGY,run_config=config,detector_config=detector_config,
               model_state_dict=model.state_dict(),optimizer_state_dict=optimizer.state_dict(),
               scaler_state_dict=scaler.state_dict(),best_retrieval_map=best,validation_metrics=metrics)
    torch.save(state,checkpoint_dir/'latest.pt')
    if improved:torch.save(state,checkpoint_dir/'best.pt')
    print(f'Epoch {epoch}: {metrics}; best mAP={best:.4f}',flush=True)
    if epoch%10==0 or epoch==NUM_EPOCHS:
        for filename,label in [('best.pt','best'),('latest.pt','latest')]:
            artifact=wandb.Artifact(f'swin-reid-{STRATEGY}-{label}',type='model',metadata={'upload_epoch':epoch})
            artifact.add_file(str(checkpoint_dir/filename));run.log_artifact(artifact)


## Final test and optional GT-box comparison
This loads the validation-selected checkpoint. Scene mode uses scene-RoI gallery embeddings, including in GT mode; crop mode re-encodes original pixel crops. Queries always use the trained query path. No optimization occurs here.


In [ ]:
state=torch.load(checkpoint_dir/'best.pt',map_location='cpu',weights_only=False)
model.load_state_dict(state['model_state_dict'],strict=True)
model.eval()
selected_epoch=state['epoch']
del state
rows=[];records_by_mode={}
for mode in ('detected','gt'):
    result=evaluate_prw(model,PRW_ROOT,device,score_threshold=SCORE_THRESHOLD,
                        detector_image_size=IMAGE_SIZE,gallery_mode=mode,gallery_embedding_mode=GALLERY_MODE,
                        return_records=True,table_queries=100,table_top_k=5)
    records_by_mode[mode]=result.pop('retrieval_records');rows.append(result)
(checkpoint_dir/'test_results.json').write_text(json.dumps(dict(checkpoint_epoch=selected_epoch,config=config,results=rows),indent=2))
try:
    scalar={f'test_{r["gallery_mode"]}/{k}':v for r in rows for k,v in r.items() if isinstance(v,(int,float))}
    scalar.update({'test/mAP':rows[0]['retrieval_mAP'],'test/top1':rows[0]['retrieval_top1'],'test/checkpoint_epoch':selected_epoch})
    run.summary.update(scalar);run.log(scalar)
    scores=wandb.Table(columns=['metric','percent'],data=[[f'{r["gallery_mode"]} {k}',100*r[k]] for r in rows for k in ('retrieval_mAP','retrieval_top1')])
    run.log({'test/final_scores':scores,'test/chart':wandb.plot.bar(scores,'metric','percent',title='Final retrieval scores')})
    import matplotlib.pyplot as plt
    for mode,records in records_by_mode.items():
        run.log({f'test_{mode}/retrievals':make_wandb_retrieval_table(records,PRW_ROOT)})
        previews=[]
        for i in range(min(10,len(records))):
            fig=plot_retrieval_query(records,PRW_ROOT,query_index=i)
            path=checkpoint_dir/f'{mode}-query-{i}.png'
            fig.savefig(path,dpi=100,bbox_inches='tight');plt.close(fig)
            previews.append(wandb.Image(str(path)))
        run.log({f'test_{mode}/images':previews})
    print('FINAL PRW TEST:',rows)
finally:run.finish()
